In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("source_version_column", "")
dbutils.widgets.text("load_type", "")
dbutils.widgets.text("landing_status", "DATA")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
source_version_column = dbutils.widgets.get("source_version_column")
load_type = dbutils.widgets.get("load_type")
landing_status = dbutils.widgets.get("landing_status")

if landing_status == "NO_DATA":
    print("[NO DATA] Skipping Silver writes; no watermark advancement is expected.")
    dbutils.notebook.exit("0")

In [0]:
print("[STEP 1] Reading source bronze table")
source_query = f"SELECT * FROM adb_caresync.bronze.{table_name} WHERE landing_timestamp = '{landing_timestamp}'"
print(f"[SOURCE QUERY]\n{source_query}")

bronze_df = spark.read.table(f"adb_caresync.bronze.{table_name}").filter(
    f"landing_timestamp='{landing_timestamp}'"
)

print(
    f"[STEP 1 COMPLETE] Source DataFrame prepared for table '{table_name}' with landing_timestamp '{landing_timestamp}'"
)
print(f"[SOURCE COLUMNS] {bronze_df.columns}")

In [0]:
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, TimestampType

print("[STEP 2] Preparing load metadata")
target_table = f"adb_caresync.silver.{table_name}"
normalized_load_type = (load_type or "FULL").strip().upper()
merge_key_list = [key.strip() for key in merge_keys.split(",") if key.strip()]
source_version_column = (source_version_column or "").strip()

print(f"[TARGET TABLE] {target_table}")
print(f"[RAW LOAD TYPE] '{load_type}'")
print(f"[NORMALIZED LOAD TYPE] {normalized_load_type}")
print(f"[MERGE KEYS] {merge_key_list}")
print(f"[SOURCE VERSION COLUMN] {source_version_column or '<not configured>'}")

def resolve_column_name(configured_name, available_columns, label):
    matches = [name for name in available_columns if name.lower() == configured_name.lower()]
    if len(matches) != 1:
        raise ValueError(
            f"{label} '{configured_name}' must resolve to exactly one source/target column; "
            f"available columns: {available_columns}"
        )
    return matches[0]

if normalized_load_type == "MERGE":
    print("[MERGE VALIDATION] Validating business keys and source version metadata")
    if not merge_key_list:
        raise ValueError("merge_keys must be provided when load_type is 'MERGE'.")
    if not source_version_column:
        raise ValueError(
            "source_version_column must be configured for MERGE. "
            "Pass the metadata watermark_column so older source versions cannot overwrite newer Silver rows."
        )

    source_lower_columns = [column.lower() for column in bronze_df.columns]
    if len(source_lower_columns) != len(set(source_lower_columns)):
        raise ValueError("Source contains column names that collide case-insensitively; MERGE is blocked.")

    merge_key_list = [
        resolve_column_name(key, bronze_df.columns, "Merge key")
        for key in merge_key_list
    ]
    source_version_column = resolve_column_name(
        source_version_column, bronze_df.columns, "Source version column"
    )

    source_version_type = bronze_df.schema[source_version_column].dataType
    if not isinstance(source_version_type, (TimestampType, DateType)):
        raise TypeError(
            f"Source version column '{source_version_column}' must be DATE or TIMESTAMP; "
            f"found {source_version_type.simpleString()}."
        )

    null_key_condition = reduce(
        lambda left, right: left | right,
        [F.col(f"`{key}`").isNull() for key in merge_key_list]
    )
    null_key_count = bronze_df.filter(null_key_condition).count()
    if null_key_count:
        raise ValueError(
            f"MERGE source contains {null_key_count} row(s) with NULL business keys "
            f"{merge_key_list}; target was not changed."
        )

    duplicate_source_keys = (
        bronze_df.groupBy(*[F.col(f"`{key}`") for key in merge_key_list])
        .count()
        .filter(F.col("count") > 1)
        .limit(10)
        .collect()
    )
    if duplicate_source_keys:
        raise ValueError(
            "MERGE source contains duplicate business keys. The batch is rejected rather "
            f"than choosing an arbitrary winner. Sample duplicate keys: {duplicate_source_keys}"
        )

    null_version_count = bronze_df.filter(F.col(f"`{source_version_column}`").isNull()).count()
    if null_version_count:
        raise ValueError(
            f"MERGE source contains {null_version_count} row(s) with NULL source version "
            f"'{source_version_column}'; target was not changed."
        )

    print("[MERGE VALIDATION COMPLETE] Source keys are non-null and unique; source version is valid.")

updated_df = (
    bronze_df.withColumn("insert_timestamp", F.current_timestamp())
    .withColumn("update_timestamp", F.current_timestamp())
)

print("[STEP 2 COMPLETE] Added insert_timestamp and update_timestamp columns")
print(f"[UPDATED COLUMNS] {updated_df.columns}")

In [0]:
print(f"[STEP 3] Starting write to target table: {target_table}")

if normalized_load_type == "FULL":
    print("[WRITE MODE] FULL")
    print(f"[WRITE ACTION] Overwriting target table {target_table}")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("[STEP 3 COMPLETE] FULL overwrite finished successfully")
elif normalized_load_type == "APPEND":
    print("[WRITE MODE] APPEND")
    print(f"[WRITE ACTION] Appending records into target table {target_table}")
    updated_df.write.mode("append").saveAsTable(target_table)
    print("[STEP 3 COMPLETE] APPEND finished successfully")
elif normalized_load_type == "MERGE":
    print("[WRITE MODE] MERGE")
    target_exists = spark.catalog.tableExists(target_table)
    print(f"[TARGET EXISTS] {target_exists}")

    if not target_exists:
        print("[MERGE FALLBACK] Target table does not exist. Creating target table with the validated source batch.")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print("[STEP 3 COMPLETE] Target table created for initial MERGE load")
    else:
        target_df = spark.table(target_table)
        target_columns = target_df.columns
        target_columns_by_lower = {column.lower(): column for column in target_columns}
        source_columns_by_lower = {column.lower(): column for column in updated_df.columns}

        if set(target_columns_by_lower) != set(source_columns_by_lower):
            missing_in_target = sorted(set(source_columns_by_lower) - set(target_columns_by_lower))
            missing_in_source = sorted(set(target_columns_by_lower) - set(source_columns_by_lower))
            raise ValueError(
                "Silver target schema does not match the incoming MERGE schema. "
                f"Missing in target: {missing_in_target}; missing in source: {missing_in_source}. "
                "Apply a reviewed schema migration before retrying."
            )

        target_merge_keys = [
            resolve_column_name(key, target_columns, "Target merge key")
            for key in merge_key_list
        ]
        target_version_column = resolve_column_name(
            source_version_column, target_columns, "Target source version column"
        )
        target_version_type = target_df.schema[target_version_column].dataType
        if target_version_type.simpleString() != source_version_type.simpleString():
            raise TypeError(
                f"Source/target version types differ: {source_version_type.simpleString()} vs "
                f"{target_version_type.simpleString()}."
            )

        target_null_key_condition = reduce(
            lambda left, right: left | right,
            [F.col(f"`{key}`").isNull() for key in target_merge_keys]
        )
        target_null_key_count = target_df.filter(target_null_key_condition).count()
        if target_null_key_count:
            raise ValueError(
                f"Existing Silver target contains {target_null_key_count} row(s) with NULL "
                f"business keys {target_merge_keys}; MERGE is blocked."
            )

        duplicate_target_keys = (
            target_df.groupBy(*[F.col(f"`{key}`") for key in target_merge_keys])
            .count()
            .filter(F.col("count") > 1)
            .limit(10)
            .collect()
        )
        if duplicate_target_keys:
            raise ValueError(
                "Existing Silver target contains duplicate business keys. Reconcile the "
                f"target before MERGE. Sample duplicate keys: {duplicate_target_keys}"
            )

        target_null_version_count = target_df.filter(
            F.col(f"`{target_version_column}`").isNull()
        ).count()
        if target_null_version_count:
            raise ValueError(
                f"Existing Silver target contains {target_null_version_count} row(s) with "
                f"NULL source version '{target_version_column}'; MERGE is blocked."
            )

        source_view = "silver_merge_source"
        updated_df.createOrReplaceTempView(source_view)
        print(f"[TEMP VIEW] Created temp view '{source_view}' for merge source")

        merge_condition = " AND ".join(
            [
                f"target.`{target_key}` = source.`{source_key}`"
                for target_key, source_key in zip(target_merge_keys, merge_key_list)
            ]
        )

        # Landing and audit timestamps are operational metadata, not business
        # payload. Equal source versions with identical business values are
        # safe replays; equal versions with changed business values are conflicts.
        technical_columns = {"landing_timestamp", "insert_timestamp", "update_timestamp"}
        payload_columns = [
            column for column in updated_df.columns
            if column.lower() not in technical_columns
        ]
        payload_differences = " OR ".join(
            [
                f"NOT (source.`{column}` <=> target.`{target_columns_by_lower[column.lower()]}`)"
                for column in payload_columns
            ]
        )

        equal_version_conflicts = []
        if payload_differences:
            conflict_query = f"""
                SELECT {", ".join([f"source.`{key}`" for key in merge_key_list])},
                       source.`{source_version_column}`
                FROM {target_table} AS target
                INNER JOIN {source_view} AS source
                    ON {merge_condition}
                WHERE source.`{source_version_column}` = target.`{target_version_column}`
                  AND ({payload_differences})
                LIMIT 10
            """
            equal_version_conflicts = spark.sql(conflict_query).collect()

        if equal_version_conflicts:
            raise ValueError(
                "Conflicting MERGE records have the same source version but different "
                "business payloads. The batch is rejected instead of applying an arbitrary "
                f"overwrite. Sample conflicts: {equal_version_conflicts}"
            )

        stale_count_query = f"""
            SELECT COUNT(*) AS stale_count
            FROM {target_table} AS target
            INNER JOIN {source_view} AS source
                ON {merge_condition}
            WHERE source.`{source_version_column}` < target.`{target_version_column}`
        """
        stale_source_count = spark.sql(stale_count_query).first()["stale_count"]
        print(
            f"[VERSION CHECK] {stale_source_count} source row(s) are older than their "
            "existing Silver versions and will not overwrite them."
        )

        update_assignments = ", ".join(
            [
                f"target.`{column}` = source.`{column}`"
                for column in updated_df.columns
                if column != "insert_timestamp"
            ]
        )
        insert_columns = ", ".join([f"`{column}`" for column in updated_df.columns])
        insert_values = ", ".join([f"source.`{column}`" for column in updated_df.columns])

        merge_sql = f"""
            MERGE INTO {target_table} AS target
            USING {source_view} AS source
            ON {merge_condition}
            WHEN MATCHED AND source.`{source_version_column}` > target.`{target_version_column}`
                THEN UPDATE SET {update_assignments}
            WHEN NOT MATCHED THEN INSERT ({insert_columns})
            VALUES ({insert_values})
        """.strip()

        print("[MERGE SQL]")
        print(merge_sql)

        spark.sql(merge_sql)
        print("[STEP 3 COMPLETE] Version-aware MERGE finished successfully")
else:
    raise ValueError("load_type must be one of: FULL, APPEND, MERGE")

In [0]:
print("[STEP 4] Computing processed record count")
record_count = updated_df.count()
print(f"[STEP 4 COMPLETE] Processed record count: {record_count}")
dbutils.notebook.exit(str(record_count))
